# 05_make_test_input - build the separate, deliberately broken test input (run once, by hand)

Takes the first 50,000 data lines of the preserved raw file (read-only), corrupts every 20th line (5%)
in five rotating ways, and saves the result to `processed/project/test_inputs/bad_v1/events.csv` with its own manifest.
The raw file and the published tables are never touched.

Expected job run with `input = test_bad`: `01_clean` succeeds and quarantines the bad lines, `02_validate` fails
`reject_rate` (5% > 1% limit), `03_publish` and `04_features` do not run.

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("lake", "amazonlake60304642")
dbutils.widgets.text("schema", "s_60304642")
dbutils.widgets.text("catalog", "dsai3202_databricks")
dbutils.widgets.text("source_container", "processed")

In [0]:
%run ./00_common

# 00_common - shared settings, paths and helpers
Every task notebook starts with `%run ./00_common`. Nothing here writes data.

| Job parameter | Meaning |
|---|---|
| `lake` | Lab-3 storage account, e.g. `amazonlake60304642` |
| `catalog`, `schema` | Unity Catalog names from Lab-5/6 |
| `run_id` | `{{job.run_id}}` in the job; a manual id when run by hand |
| `input` | `full` = the preserved raw file, `test_bad` = the separate corrupted test input |
| `source_container` | `raw` (default) or `processed` if Databricks cannot read raw (landing copy) |
| `compare_to` | a successful run id; the run then proves SAME CONTENT against it |

Adapted from DSAI3202 Lab-5 (validation, balanced counts, Delta) and Lab-6 (prepare -> validate -> publish, run folders, gate).

{
  "lake": "amazonlake60304642",
  "catalog": "dsai3202_databricks",
  "schema": "s_60304642",
  "run_id": "manual-20261114T124727Z",
  "input": "full",
  "source_container": "processed",
  "compare_to": "",
  "silver_table": "dsai3202_databricks.s_60304642.project_silver_events",
  "features_table": "dsai3202_databricks.s_60304642.project_features_v1"
}


/home/spark-179888ce-f62e-4060-95f9-d7/.ipykernel/2498/command-5296352083289421-820008573:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  P["run_id"] = "manual-" + datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


In [0]:
import hashlib
P_input_saved = P["input"]; P["input"] = "full"
src, _ = input_location(); P["input"] = P_input_saved
text = dbutils.fs.head(src, 9_000_000)
lines = text.split("\n")[:-1]                 # drop a possibly cut last line
header, rows = lines[0], lines[1:50_001]
assert header.split(",")[:2] == ["event_time", "event_type"], header

def corrupt(i, line):
    f = line.split(",")
    kind = (i // 20) % 5
    if kind == 0: f[1] = "refund"                       # bad_event_type
    elif kind == 1: f[6] = "-1.00"                       # bad_price
    elif kind == 2: f[7] = ""                            # missing_key
    elif kind == 3: f[0] = "not-a-time"                  # bad_timestamp
    else: f = f + ["extra", "fields"]                    # malformed_row
    return ",".join(f)

out = [header] + [corrupt(i, r) if i % 20 == 0 else r for i, r in enumerate(rows)]
body = "\n".join(out) + "\n"
dbutils.fs.put(f"{TEST_INPUT}/events.csv", body, overwrite=True)
manifest = {"purpose": "controlled quality failure test input (not real data quality)",
            "derived_from": src, "data_rows": len(rows), "corrupted_rows": len([i for i in range(len(rows)) if i % 20 == 0]),
            "corruption": "every 20th line; rotating bad_event_type, bad_price, missing_key, bad_timestamp, malformed_row",
            "file": {"name": "events.csv", "data_rows": len(rows), "sha256": hashlib.sha256(body.encode()).hexdigest()}}
write_json(f"{TEST_INPUT}/manifest.json", manifest)
print(json.dumps(manifest, indent=2))

[Truncated to first 9000000 bytes]
Wrote 6643016 bytes.
Wrote 534 bytes.
saved abfss://processed@amazonlake60304642.dfs.core.windows.net/project/test_inputs/bad_v1/manifest.json
{
  "purpose": "controlled quality failure test input (not real data quality)",
  "derived_from": "abfss://processed@amazonlake60304642.dfs.core.windows.net/project/landing/rees46/2019-Nov/sha256=addd9a27ed99/2019-Nov.csv",
  "data_rows": 50000,
  "corrupted_rows": 2500,
  "corruption": "every 20th line; rotating bad_event_type, bad_price, missing_key, bad_timestamp, malformed_row",
  "file": {
    "name": "events.csv",
    "data_rows": 50000,
    "sha256": "4b6d67bdcf039a9ded9052c48f0a2fbb049216422bfd8a85b7de7935fc9c9265"
  }
}
